# Social IQa Multi-Agent Test (Qwen3-4B)

## Setup
- Cell 1: pip install + model load (run once)
- Cell 2: Social IQa loader + agents
- Cell 3: Description
- Cell 4: Experiment (4 conditions x 7 budgets)

---

## API Reference

### Agent creation
```python
agent = Agent("name", "system prompt")
```

### Inference
```python
r = agent.say("message", max_tokens=256, thinking=False)
r["response"]   # response text
r["tokens"]     # output token count
r["time"]       # elapsed time (sec)
```

In [ ]:
# ============================================================
# Cell 1: Model load (run once)
# ============================================================
!pip install -q torch transformers accelerate datasets

import torch
import time
import re

# Global model/tokenizer
_model = None
_tokenizer = None
_device = None

def load_model(model_id: str = "Qwen/Qwen3-4B"):
    global _model, _tokenizer, _device
    from transformers import AutoModelForCausalLM, AutoTokenizer
    _device = "cuda" if torch.cuda.is_available() else "cpu"
    dtype = torch.float16 if _device == "cuda" else torch.float32
    print(f"Device: {_device}")
    if _device == "cuda":
        print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Loading {model_id}...")
    t0 = time.time()
    _tokenizer = AutoTokenizer.from_pretrained(model_id)
    _model = AutoModelForCausalLM.from_pretrained(
        model_id, torch_dtype=dtype,
        device_map="auto" if _device == "cuda" else None,
    )
    if _device == "cpu":
        _model = _model.to(_device)
    params = sum(p.numel() for p in _model.parameters()) / 1e9
    print(f"Loaded in {time.time()-t0:.1f}s ({params:.1f}B params)")

class Agent:
    def __init__(self, name: str, system_prompt: str):
        self.name = name
        self.system_prompt = system_prompt
        self.history = []

    def say(self, message: str, max_tokens: int = 256, thinking: bool = False) -> dict:
        if _model is None:
            raise RuntimeError("Run load_model() first.")
        messages = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": message},
        ]
        text = _tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
            enable_thinking=thinking
        )
        inputs = _tokenizer(text, return_tensors="pt").to(_model.device)
        t0 = time.time()
        with torch.no_grad():
            output = _model.generate(**inputs, max_new_tokens=max_tokens, do_sample=False)
        elapsed = time.time() - t0
        response = _tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        gen_tokens = output.shape[1] - inputs["input_ids"].shape[1]
        result = {"response": response, "tokens": gen_tokens, "time": round(elapsed, 1)}
        self.history.append({"input": message, **result})
        return result

    def set_prompt(self, new_prompt: str):
        self.system_prompt = new_prompt
        return self

    def __repr__(self):
        return f"Agent('{self.name}')"

def send(sender, receiver, message, max_tokens=256, verbose=True):
    s = sender.say(message, max_tokens=max_tokens)
    r = receiver.say(s["response"], max_tokens=max_tokens)
    if verbose:
        print(f"[{sender.name}] {s['tokens']}tok, {s['time']}s")
        print(f"  >> {s['response'][:200]}")
        print(f"[{receiver.name}] {r['tokens']}tok, {r['time']}s")
        print(f"  >> {r['response'][:200]}")
    return {"sender": s, "receiver": r, "tx_tokens": s["tokens"], "total_tokens": s["tokens"] + r["tokens"]}

def chain(agents, message, max_tokens=256, verbose=True):
    current = message
    results = []
    for agent in agents:
        r = agent.say(current, max_tokens=max_tokens)
        results.append({"agent": agent.name, **r})
        if verbose:
            print(f"[{agent.name}] {r['tokens']}tok, {r['time']}s")
            print(f"  >> {r['response'][:200]}")
        current = r["response"]
    return {"steps": results, "final": results[-1]["response"], "total_tokens": sum(r["tokens"] for r in results)}

def extract_number(text):
    nums = re.findall(r'-?[\d,]+\.?\d*', text.replace(',', ''))
    return float(nums[0]) if nums else -999

def grade(got, expected, tolerance=0.1):
    if expected == 0:
        return abs(got) < tolerance
    return abs(got - expected) / abs(expected) < tolerance

load_model("Qwen/Qwen3-4B")

In [ ]:
# ============================================================
# Cell 2: Social IQa loader + Agents
# ============================================================
import random
import re as _re
from datasets import load_dataset

summarizer = Agent("Summarizer", "You are a Summarizer agent.")
answerer = Agent("Answerer", "You are an Answerer agent.")

print("Loading Social IQa...")
siqa = load_dataset("social_i_qa", split="validation", trust_remote_code=True)
LABEL_MAP = {"1": "A", "2": "B", "3": "C"}

def format_siqa(ex):
    choices_text = f"A) {ex['answerA']}\nB) {ex['answerB']}\nC) {ex['answerC']}"
    return {
        "context": ex["context"],
        "question": ex["question"],
        "choices_text": choices_text,
        "answer": LABEL_MAP[ex["label"]],
    }

def sample_siqa(n, seed=42):
    random.seed(seed)
    samples = random.sample(list(siqa), min(n, len(siqa)))
    return [format_siqa(s) for s in samples]

def extract_answer(response):
    text = response.strip()
    if text.upper() in ("A", "B", "C"):
        return text.upper()
    m = _re.search(r'(?:answer|choice)[\s:is]*([A-C])\b', text, _re.IGNORECASE)
    if m: return m.group(1).upper()
    m = _re.search(r'^([A-C])[)\.]', text, _re.MULTILINE)
    if m: return m.group(1)
    m = _re.search(r'\b([A-C])\b', text)
    if m: return m.group(1)
    return "N/A"

print(f"Social IQa loaded: {len(siqa)} questions")

## Social IQa 4-Condition Experiment

**Dataset:** Social IQa (validation split, 3 choices A/B/C)

**4 Conditions (2x2 design):**
- `blind`: A_BLIND + B_BLIND (neither agent sees question/choices)
- `a_aware`: A_AWARE + B_BLIND (A sees question+choices, B does not know)
- `b_aware`: A_BLIND + B_AWARE (A is blind, B knows A's encoding scheme)
- `mutual`: A_AWARE + B_AWARE (both agents are aware)

**Token budgets:** 16, 24, 32, 40, 48, 52, 64

**A caching:** blind and choices outputs cached per question per budget (2 A calls instead of 4)

In [ ]:
# ============================================================
# Cell 4: Social IQa 4-Condition Experiment
# ============================================================

# --- Exact prompts ---
A_BLIND = """You are A.

Read the context only.

Encode the situation into exactly three short fragments separated by " | " in this fixed order:
1) main event
2) likely motive / intent before the event
3) likely reaction / next consequence after the event

Each fragment must be short.
Do not explain.
Output only the three fragments."""

A_AWARE = """You are A.

Read the context, question, and answer choices.

Use the same 3-fragment code:
1) main event
2) likely motive / intent
3) likely reaction / next consequence

Make the fragment most relevant to the question the most specific and choice-discriminative.
Keep the other two generic and short.
Do not explain.
Output only the three fragments."""

B_BLIND = """You are B.

You will get:
- a 3-fragment summary from A
- the question
- the answer choices

Choose the best answer.

Output only A, B, or C."""

B_AWARE = """You are B.

A always uses this fixed 3-fragment code:
1) main event
2) motive / intent before the event
3) reaction / next consequence after the event

Use the fragment that matches the question type as primary evidence:
- why / wanted / needed / intent -> fragment 2
- how feel / reaction / what happens next -> fragment 3
- what happened / what did X do -> fragment 1

Use the other fragments only as backup.

Output only A, B, or C."""

# --- Conditions ---
CONDITIONS = {
    "blind":   {"a_type": "blind",   "b_knows": False},
    "a_aware": {"a_type": "choices", "b_knows": False},
    "b_aware": {"a_type": "blind",   "b_knows": True},
    "mutual":  {"a_type": "choices", "b_knows": True},
}
TX_BUDGETS = [16, 24, 32, 40, 48, 52, 64]

# --- Sample questions ---
Q1 = sample_siqa(30, seed=42)
print(f"Social IQa 4-COND: {len(Q1)} questions, budgets={TX_BUDGETS}")

# --- Run experiment ---
results = {b: {c: [] for c in CONDITIONS} for b in TX_BUDGETS}

for qi, q in enumerate(Q1):
    if (qi + 1) % 5 == 0:
        print(f"  Q{qi+1}/{len(Q1)}...")
    for budget in TX_BUDGETS:
        # A caching: 2 calls instead of 4
        a_cache = {}

        # A_BLIND
        summarizer.set_prompt(A_BLIND)
        a_cache["blind"] = summarizer.say(
            f"Context:\n{q['context']}",
            max_tokens=budget
        )

        # A_AWARE
        summarizer.set_prompt(A_AWARE)
        a_cache["choices"] = summarizer.say(
            f"Context:\n{q['context']}\n\nQuestion: {q['question']}\nChoices:\n{q['choices_text']}",
            max_tokens=budget
        )

        for cond, cfg in CONDITIONS.items():
            a_out = a_cache[cfg["a_type"]]
            b_sys = B_AWARE if cfg["b_knows"] else B_BLIND
            answerer.set_prompt(b_sys)
            b_out = answerer.say(
                f"Summary:\n{a_out['response']}\n\nQuestion: {q['question']}\nChoices:\n{q['choices_text']}\n\nAnswer:",
                max_tokens=48
            )
            ans = extract_answer(b_out["response"])
            results[budget][cond].append({
                "correct": ans == q["answer"],
                "a_tokens": a_out["tokens"],
                "b_tokens": b_out["tokens"],
            })

# --- Results table ---
print(f"\n{'Budget':<8} {'blind':<15} {'a_aware':<15} {'b_aware':<15} {'mutual':<15}")
print("-" * 68)
for b in TX_BUDGETS:
    row = f"{b}tok  "
    for c in CONDITIONS:
        res = results[b][c]
        acc = sum(r["correct"] for r in res) / len(res)
        avg_t = sum(r["a_tokens"] for r in res) / len(res)
        row += f"{acc*100:>4.0f}% (avg{avg_t:.0f}t) "
    print(row)

# --- 2x2 Interaction ---
print(f"\n--- 2x2 Interaction ---")
for b in TX_BUDGETS:
    bl = sum(r["correct"] for r in results[b]["blind"]) / 30
    aa = sum(r["correct"] for r in results[b]["a_aware"]) / 30
    ba = sum(r["correct"] for r in results[b]["b_aware"]) / 30
    mu = sum(r["correct"] for r in results[b]["mutual"]) / 30
    a_eff = ((aa - bl) + (mu - ba)) / 2
    b_eff = ((ba - bl) + (mu - aa)) / 2
    print(f"  @{b}tok: A={a_eff*100:+.1f}%p B={b_eff*100:+.1f}%p mutual={mu*100:.0f}% blind={bl*100:.0f}%")